# Register Model

# Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Model Service Registration to MLFlow

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 21 ms, sys: 14.6 ms, total: 35.6 ms
Wall time: 1.14 s


In [2]:
MIN_TOTAL_RAM_GB = 16
MIN_TOTAL_VRAM_GB = 8


from ai_studio_blueprint_kit.memory_guard import run_memory_check_notebook


run_memory_check_notebook(
    min_total_ram_gb=MIN_TOTAL_RAM_GB,
    min_total_vram_gb=MIN_TOTAL_VRAM_GB,
)

# Start Execution

In [3]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [4]:
start_time = time.time()  

logger.info("Notebook execution started.")

2026-04-16 22:01:20 - INFO - Notebook execution started.


# Install and Import Libraries

In [5]:
import os
import sys
import logging

# Define the relative path to the 'src' directory (two levels up from current working directory)
src_path = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add 'src' directory to system path for module imports (e.g., utils)
if src_path not in sys.path:
    sys.path.append(src_path)

# === Standard Library Imports ===
import os
import sys
import logging
import json
import time
import warnings
from datetime import datetime
from pathlib import Path

# === Third-Party Imports ===
import numpy as np
import pandas as pd
import webvtt
import mlflow
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
from operator import itemgetter

# === MLflow Signature Imports ===
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

# === Project-Specific Imports (from src.utils) ===
from src.utils import (
    load_config,
    load_secrets,
    load_secrets_to_env,
    configure_proxy,
    initialize_llm,
    configure_hf_cache
)
from src.prompt_templates import format_chunk_summarization_prompt

# === Import Logger ===
from src.mlflow import Logger

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from IPython import get_ipython

# Configure Settings

In [6]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

In [7]:
# In case you just want to run this cell without the rest of the notebook 
# (you still need to install the requirements and run the import block), run the following block:
CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"
# Define demo folder path
DEMO_FOLDER = "../demo"

In [8]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

No secrets file found at ../configs/secrets.yaml; relying on preexisting environment
✅ Configuration loaded successfully
✅ Secrets loaded successfully


## Model Service Registration

In this example, we illustrate a different approach to create a text summarizer. Instead of splitting the text into topics and summarize the topics individually, this model service provides a REST API endpoint to allow summarization of an entire text, in a single call to the model.

## Text Summarization Service

This section demonstrates how to use our SummarizationService from the core directory. This approach improves code organization by separating the service implementation from the notebook, making it easier to maintain and update.

In [9]:
# Create MLflow model signature for text summarization
# Define input schema: expects text to be summarized
input_schema = Schema([
    ColSpec("string", "text")
])

# Define output schema: returns the summary
output_schema = Schema([
    ColSpec("string", "summary")
])

# Create the signature
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

logger.info("✅ Model signature created successfully")

2026-04-16 22:01:27 - INFO - ✅ Model signature created successfully


In [10]:
%%time

mlflow.set_tracking_uri('/phoenix/mlflow')
# Set up the MLflow experiment
mlflow.set_experiment("Summarization_Service")

# === Get model path from config ===
model_path = config.get("model_path")
if model_path and os.path.exists(model_path):
    logger.info(f"✅ Model file found at: {model_path}")
else:
    logger.info(f"⚠️ Warning: Model file not found at {model_path}. Please verify the path in config.yaml.")

logger.info('Starting the experiment: Summarization_Service')
logger.info(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")

# Use the Logger's log_model method to register the model in MLflow
with mlflow.start_run(run_name="Text_Summarization_Service") as run:
    # Log and register the model using the service's classmethod with signature
    Logger.log_model(
        artifact_path="text_summarization_service",
        secrets_dict=secrets if secrets else None,
        config_path=CONFIG_PATH,
        model_path=model_path,
        demo_folder=DEMO_FOLDER,
        signature=signature
    )
    
    # Register the model in MLflow Model Registry
    model_uri = f"runs:/{run.info.run_id}/text_summarization_service"
    mlflow.register_model(model_uri=model_uri, name="Text_Summarization_Service")
    print(f"Model registered successfully with run ID: {run.info.run_id}")

2026/04/16 22:01:27 INFO mlflow.tracking.fluent: Experiment with name 'Summarization_Service' does not exist. Creating a new experiment.
2026-04-16 22:01:27 - INFO - ✅ Model file found at: /home/jovyan/datafabric/meta-llama3.1-8b-Q8/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf
2026-04-16 22:01:27 - INFO - Starting the experiment: Summarization_Service
2026-04-16 22:01:27 - INFO - Using MLflow tracking URI: /phoenix/mlflow
Successfully registered model 'Text_Summarization_Service'.
2026/04/16 22:05:56 WARNING mlflow.tracking._model_registry.fluent: Run with id bdf38c61566a477eb64a96e936056a61 has no artifacts at artifact path 'text_summarization_service', registering model based on models:/m-8847c446b28940ce8d0713d3e5a0f0d1 instead
Created version '1' of model 'Text_Summarization_Service'.


Model registered successfully with run ID: bdf38c61566a477eb64a96e936056a61
CPU times: user 1.34 s, sys: 28.5 s, total: 29.8 s
Wall time: 4min 29s


In [11]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

2026-04-16 22:05:57 - INFO - ⏱️ Total execution time: 4m 36.79s


In [1]:
status = "Notebook execution completed successfully"
print(f"Message: {status}")

Message: Notebook execution completed successfully


In [ ]:
app = get_ipython()
app.kernel.do_shutdown(restart=False)

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).